<img src="../img/viu_logo.png" width="200">

# Aprendizaje por Refuerzo: Proyecto de programación

### Desarrollando un agente DQN para el entorno Enduro-v0

<img src="https://ale.farama.org/_images/enduro.gif" width="300">

- **Profesor:** Francisco Muñoz Montoya
- **Autores:** Guillermo Sanchez Garcia, Diego Aguado Sanchez, Alejandro Pequeno Lizcano, Jaume Montanera

### Índice
0. [Introducción](#0-introducción)
1. [Implementación base](#1-implementación-base)
2. [Mejoras y experimentos](#2-mejoras-y-experimentos)
3. [Conclusiones](#3-conclusiones)

## 0. Introducción

Consideraciones a tener en cuenta:

- El entorno sobre el que trabajaremos es **Enduro-v0** y el algoritmo que usaremos será _DQN_.

- El objetivo es que el agente **supere el primer día de carrera** (≥ 200 coches adelantados) en **100 episodios consecutivos** en modo test.

- El requisito mínimo será alcanzado cuando el agente consiga una **media de recompensa por encima de los puntos indicados en el listado por grupos en modo test**. Por ello, esta media de la recompensa se calculará a partir del código de test en la última celda del notebook. En nuestro caso, el entorno Enduro-v0, el requisito mínimo es superar el primer día de carrera durante 100 episodios consecutivos.

Este proyecto práctico consta de tres partes:

1.   Implementar la red neuronal que se usará en la solución
2.   Implementar las distintas piezas de la solución DQN y probar al menos 3 propuestas diferentes de mejora.
3.   Justificar la respuesta en relación a los resultados obtenidos e incluir al menos 3 gráficas relevantes comparando las 3 propuestas.

**Rúbrica**: Se valorará la originalidad en la solución aportada, así como la capacidad de discutir los resultados de forma detallada. El requisito mínimo servirá para aprobar la actividad, bajo premisa de que la discusión del resultado sera apropiada.

IMPORTANTE:

* Si no se consigue una puntuación óptima, responder sobre la mejor puntuación obtenida.
* Para entrenamientos largos, recordad que podéis usar checkpoints de vuestros modelos para retomar los entrenamientos. En este caso, recordad cambiar los parámetros adecuadamente (sobre todo los relacionados con el proceso de exploración).
* Se deberá entregar unicamente el notebook y los pesos del mejor modelo en un fichero .zip, de forma organizada.
* Cada alumno deberá de subir la solución de forma individual.

## Imports, configuración y entorno

Importaciones, shim de compatibilidad keras-rl2 ↔ TF 2.18 y carga de variables de entorno.

In [1]:
from __future__ import division

# Librerías estándar
# ===========================================================
import os
import sys
import types
import platform
import random
from pathlib import Path
import numpy as np

# Variables de entorno
# ===========================================================
from dotenv import load_dotenv

load_dotenv()

# keras-rl2 usa tensorflow.keras internamente: debe establecerse antes de importar TF
# ===========================================================
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# Deep learning: TF
# ===========================================================
import tensorflow as tf

# Keras 2 (tf-keras): modelos, capas, optimizadores
# ===========================================================
import tf_keras
from tf_keras.models import Sequential
from tf_keras.layers import Dense, Flatten, Convolution2D, Permute
from tf_keras.optimizers.legacy import Adam
import tf_keras.backend as K

# Shim de compatibilidad: keras-rl2 1.0.5 usa tensorflow.python.keras (eliminado en TF 2.16+)
# ===========================================================
import tensorflow.keras as _tfkeras

if not hasattr(_tfkeras, "__version__"):
    _tfkeras.__version__ = tf_keras.__version__

_compat_generic = types.ModuleType("tensorflow.python.keras.utils.generic_utils")
_compat_generic.Progbar = tf_keras.utils.Progbar

_compat_utils = types.ModuleType("tensorflow.python.keras.utils")
_compat_utils.generic_utils = _compat_generic

_compat_keras = types.ModuleType("tensorflow.python.keras")
_compat_keras.callbacks = tf_keras.callbacks
_compat_keras.utils = _compat_utils

sys.modules.setdefault("tensorflow.python.keras", _compat_keras)
sys.modules.setdefault("tensorflow.python.keras.callbacks", tf_keras.callbacks)
sys.modules.setdefault("tensorflow.python.keras.utils", _compat_utils)
sys.modules.setdefault("tensorflow.python.keras.utils.generic_utils", _compat_generic)

# Entorno RL
# ===========================================================
import gym
from PIL import Image

# keras-rl2: agente DQN, políticas, memoria, callbacks
# ===========================================================
from rl.agents.dqn import DQNAgent
from rl.policy import LinearAnnealedPolicy, BoltzmannQPolicy, EpsGreedyQPolicy
from rl.memory import SequentialMemory
from rl.core import Processor
from rl.callbacks import Callback, FileLogger, ModelIntervalCheckpoint

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
IS_ARM_MAC = sys.platform == "darwin" and platform.machine() == "arm64"

print("TensorFlow version:", tf.__version__)
print("tf-keras version:  ", tf_keras.__version__)
print("ARM Mac:           ", IS_ARM_MAC)
print("GPUs disponibles:  ", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.18.0
tf-keras version:   2.18.0
ARM Mac:            True
GPUs disponibles:   [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Constantes

Todos los hiperparámetros y rutas centralizados. Modificar aquí para experimentar.

| Grupo | Constantes clave |
|---|---|
| Entorno | `ENV_NAME`, `FRAME_H/W`, `CROP_BOTTOM`, `SHAPE`, `WINDOW_LENGTH` |
| Red | `DENSE_UNITS`, `LEARNING_RATE` |
| Replay | `MEMORY_LIMIT` |
| Política | `EPS_MAX/MIN/TEST`, `EPS_ANNEALING_STEPS` |
| Agente | `GAMMA`, `TARGET_MODEL_UPDATE`, `TRAIN_INTERVAL`, `DELTA_CLIP` |
| Entrenamiento | `NB_STEPS`, `CHECKPOINT_INTERVAL` |
| Test | `NB_TEST_EPISODES`, `DAY1_TARGET` |

In [3]:
# Paths
# ===========================================================
BASE_FOLDER = Path(os.getenv("BASE_FOLDER"))
WEIGHTS_FOLDER = Path(os.getenv("WEIGHTS_FOLDER"))

# Reproducibilidad
# ===========================================================
SEED = int(os.getenv("SEED", 42))
np.random.seed(SEED)

# Entorno
# ===========================================================
ENV_NAME = "Enduro-v0"  # nombre del entorno: se usa en todo el código (pesos, logs, etc.)
# Enduro raw frame: (210, 160, 3) RGB: se escala a SHAPE en el processor
FRAME_H = 210
FRAME_W = 160
CROP_BOTTOM = 16  # píxeles del HUD inferior de Enduro a eliminar antes de escalar
SHAPE = (84, 84)  # resolución de entrada estándar DQN (post-crop y resize)
WINDOW_LENGTH = 4  # frames apilados para capturar movimiento
INPUT_SHAPE = (WINDOW_LENGTH,) + SHAPE

# Enduro: objetivo del primer día
# ===========================================================
DAY1_TARGET = 200  # coches que hay que adelantar para completar el día 1

# Red neuronal
# ===========================================================
DENSE_UNITS = 512
LEARNING_RATE = 0.00025

# Memoria replay
# ===========================================================
MEMORY_LIMIT = 1_000_000  # estándar DQN Atari

# Política eps-greedy con annealing
# ===========================================================
EPS_MAX = 1.0
EPS_MIN = 0.1
EPS_TEST = 0.05
EPS_ANNEALING_STEPS = 1_000_000  # eps decae de EPS_MAX a EPS_MIN durante estos pasos

# Agente DQN
# ===========================================================
NB_STEPS_WARMUP = 50_000  # pasos de exploración pura antes de empezar a aprender
GAMMA = 0.99  # factor de descuento (Enduro tiene horizonte largo)
TARGET_MODEL_UPDATE = 10_000  # cada cuántos pasos se sincroniza la target network
TRAIN_INTERVAL = 4  # actualizar pesos cada N pasos de entorno
DELTA_CLIP = 1.0  # Huber loss clipping

# Entrenamiento
# ===========================================================
NB_STEPS = 2_000_000  # número total de pasos de entrenamiento (entorno)
CHECKPOINT_INTERVAL = 100_000  # cada cuántos pasos se guarda un checkpoint de pesos
LOG_INTERVAL = 1_000  # cada cuántos pasos se loguea información de entrenamiento
LOG_DISPLAY_INTERVAL = 10_000  # cada cuántos pasos se loguea información de entrenamiento en pantalla

# Test
# ===========================================================
NB_TEST_EPISODES = 100  # objetivo: superar el primer día de carrera en 100 episodios consecutivos

# Inicializar entorno
# ===========================================================
env = gym.make(ENV_NAME)
env.seed(SEED)
nb_actions = env.action_space.n
print(f"Entorno: {ENV_NAME} | Acciones: {nb_actions} | Frame bruto: ({FRAME_H}, {FRAME_W}, 3)")


Entorno: Enduro-v0 | Acciones: 9 | Frame bruto: (210, 160, 3)


/Users/alejandro/miniconda3/envs/ar_env/lib/python3.10/site-packages/gym/envs/registration.py:593: UserWarning: WARN: The environment Enduro-v0 is out of date. You should consider upgrading to version `v4`.
  logger.warn(
A.L.E: Arcade Learning Environment (version 0.7.5+db37282)
[Powered by Stella]
/Users/alejandro/miniconda3/envs/ar_env/lib/python3.10/site-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/Users/alejandro/miniconda3/envs/ar_env/lib/python3.10/site-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


## 1. Implementación base

### 1.1 Implementación de la red neuronal

Arquitectura **DeepMind DQN** (Mnih et al. 2015), estándar para Atari a 84×84

In [4]:
class AtariProcessor(Processor):
    """Preprocesado estándar DQN adaptado a Enduro-v0.

    Pipeline por frame:
      1. Crop inferior (CROP_BOTTOM px): elimina el HUD de puntuación de Enduro.
      2. Resize a SHAPE (84x84) con antialiasing.
      3. Conversión a escala de grises: reduce canales de 3 a 1 manteniendo contraste.
      4. Normalización [0, 1] en process_state_batch (sobre el stack de 4 frames).
      5. Reward clipping a [-1, 1]: estabiliza el entrenamiento (reward máx. Enduro = +1/coche).
    """

    def process_observation(self, observation):
        # Enduro raw frame: (210, 160, 3) uint8
        assert observation.ndim == 3 and observation.shape == (FRAME_H, FRAME_W, 3)
        img = Image.fromarray(observation)
        # Crop HUD inferior antes de escalar para no perder información de la carretera
        img = img.crop((0, 0, FRAME_W, FRAME_H - CROP_BOTTOM))
        img = img.resize(SHAPE, Image.BILINEAR).convert("L")
        processed = np.array(img)
        assert processed.shape == SHAPE
        return processed.astype("uint8")

    def process_state_batch(self, batch):
        # batch shape: (B, WINDOW_LENGTH, H, W) -> normalizar a [0, 1]
        return batch.astype("float32") / 255.0

    def process_reward(self, reward):
        # Enduro: +1 por coche adelantado, 0 resto -> clip no pierde información
        return np.clip(reward, -1.0, 1.0)


In [5]:
# Arquitectura DeepMind (Mnih et al. 2015)
# Entrada: INPUT_SHAPE = (4, 84, 84): stack de 4 frames preprocesados
# Permute adapta channels_first -> channels_last para aprovechar los kernels cuDNN/Metal de TF
# Salida: nb_actions = 9 (NOOP, FIRE, RIGHT, LEFT, DOWN, DOWNRIGHT, DOWNLEFT, RIGHTFIRE, LEFTFIRE)
model = Sequential()
if K.image_data_format() == "channels_last":
    model.add(Permute((2, 3, 1), input_shape=INPUT_SHAPE))
elif K.image_data_format() == "channels_first":
    model.add(Permute((1, 2, 3), input_shape=INPUT_SHAPE))
else:
    raise RuntimeError("image_data_format desconocido.")

model.add(Convolution2D(32, (8, 8), strides=(4, 4), activation="relu"))
model.add(Convolution2D(64, (4, 4), strides=(2, 2), activation="relu"))
model.add(Convolution2D(64, (3, 3), strides=(1, 1), activation="relu"))
model.add(Flatten())
model.add(Dense(DENSE_UNITS, activation="relu"))
model.add(Dense(nb_actions, activation="linear"))

print(model.summary())


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 permute (Permute)           (None, 84, 84, 4)         0         
                                                                 
 conv2d (Conv2D)             (None, 20, 20, 32)        8224      
                                                                 
 conv2d_1 (Conv2D)           (None, 9, 9, 64)          32832     
                                                                 
 conv2d_2 (Conv2D)           (None, 7, 7, 64)          36928     
                                                                 
 flatten (Flatten)           (None, 3136)              0         
                                                                 
 dense (Dense)               (None, 512)               1606144   
                                                                 
 dense_1 (Dense)             (None, 9)                 4

### 1.2 Implementación del agente (DQN)

Componentes del agente:

- **`AtariProcessor`**: preprocesa el frame bruto de Enduro `(210, 160, 3)` a `(84, 84)` gris,
  recorta el HUD inferior y normaliza a [0, 1].
- **`SequentialMemory`**: replay buffer de 1 M transiciones.
- **`LinearAnnealedPolicy`**: epsilon-greedy con annealing lineal de 1.0 -> 0.1 en 1 M pasos.

In [6]:
processor = AtariProcessor()

memory = SequentialMemory(limit=MEMORY_LIMIT, window_length=WINDOW_LENGTH)

policy = LinearAnnealedPolicy(
    EpsGreedyQPolicy(),
    attr="eps",
    value_max=EPS_MAX,
    value_min=EPS_MIN,
    value_test=EPS_TEST,
    nb_steps=EPS_ANNEALING_STEPS,
)

In [7]:
dqn = DQNAgent(
    model=model,
    nb_actions=nb_actions,
    policy=policy,
    memory=memory,
    processor=processor,
    nb_steps_warmup=NB_STEPS_WARMUP,
    gamma=GAMMA,
    target_model_update=TARGET_MODEL_UPDATE,
    train_interval=TRAIN_INTERVAL,
    delta_clip=DELTA_CLIP,
)
dqn.compile(Adam(learning_rate=LEARNING_RATE), metrics=["mae"])


2026-06-28 23:57:29.746414: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5 Pro
2026-06-28 23:57:29.746438: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 64.00 GB
2026-06-28 23:57:29.746443: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 25.92 GB
I0000 00:00:1782683849.746454 4220352 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1782683849.746470 4220352 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
I0000 00:00:1782683849.749768 4220352 mlir_graph_optimization_pass.cc:401] MLIR V1 optimization pass is not enabled
2026-06-28 23:57:29.751615: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is

In [8]:
class SaveBestWeights(Callback):
    """Guarda en RAM los pesos del mejor episodio; al terminar los escribe a disco.
    Evita problemas de formato de archivo (TF checkpoint vs HDF5).
    """

    def __init__(self, dirpath, env_name, verbose=1):
        super().__init__()
        self._dirpath = Path(dirpath)
        self._env_name = env_name
        self.verbose = verbose
        self.best_reward = -np.inf
        self._best_step = 0
        self._best_weights = None

    def on_episode_end(self, episode, logs):
        reward = logs.get("episode_reward", -np.inf)
        if reward > self.best_reward:
            prev = self.best_reward
            self.best_reward = reward
            self._best_step = self.model.step
            self._best_weights = self.model.model.get_weights()
            if self.verbose:
                print(f"\n[Best] Ep {episode} | step {self._best_step} | reward {reward:.1f} (prev {prev:.1f})")

    def on_train_end(self, logs):
        if self._best_weights is not None:
            self.model.model.set_weights(self._best_weights)
            final_path = str(self._dirpath / f"dqn_{self._env_name}_best_step{self._best_step}.h5f")
            self.model.save_weights(final_path, overwrite=True)
            if self.verbose:
                print(f"\n[Best] Guardado en: {final_path} | reward {self.best_reward:.1f}")


In [ ]:
# Crear estructura de carpetas
# ===========================================================
checkpoints_dir = WEIGHTS_FOLDER / "checkpoints"
logs_dir = WEIGHTS_FOLDER / "logs"
final_dir = WEIGHTS_FOLDER / "final"
for d in [checkpoints_dir, logs_dir, final_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Rutas de archivos
# ===========================================================
checkpoint_file = str(checkpoints_dir / f"dqn_{ENV_NAME}_step{{step}}.h5f")
log_file = str(logs_dir / f"dqn_{ENV_NAME}_log.json")

callbacks = [
    ModelIntervalCheckpoint(checkpoint_file, interval=CHECKPOINT_INTERVAL),
    FileLogger(log_file, interval=LOG_INTERVAL),
    SaveBestWeights(final_dir, ENV_NAME),
]

dqn.fit(env, nb_steps=NB_STEPS, callbacks=callbacks, log_interval=LOG_DISPLAY_INTERVAL, visualize=False)

final_file = str(final_dir / f"dqn_{ENV_NAME}_final_step{dqn.step}.h5f")
dqn.save_weights(final_file, overwrite=True)
print(f"Pesos finales: {final_file}")

Training for 2000000 steps ...


/Users/alejandro/miniconda3/envs/ar_env/lib/python3.10/site-packages/tf_keras/src/engine/training_v1.py:2354: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,
2026-06-28 15:18:29.476332: W tensorflow/c/c_api.cc:305] Operation '{name:'dense_1/BiasAdd' id:125 op device:{requested: '', assigned: ''} def:{{{node dense_1/BiasAdd}} = BiasAdd[T=DT_FLOAT, _has_manual_control_dependencies=true, data_format="NHWC"](dense_1/MatMul, dense_1/BiasAdd/ReadVariableOp)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
2026-06-28 15:18:29.483948: W tensorflow/c/c_api.cc:305] Operation '{name:'total/Assign' id:360 op device:{requested: '', assigned: ''} def:{{{node total/Assign}} = AssignVariableOp[_ha

Interval 1 (0 steps performed)
   45/10000 [..............................] - ETA: 23s - reward: 0.0000e+00

/Users/alejandro/miniconda3/envs/ar_env/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:227: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the environment with new step API. 
  logger.deprecation(
/Users/alejandro/miniconda3/envs/ar_env/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(done, (bool, np.bool8)):


 4453/10000 [============>.................] - ETA: 11s - reward: 0.0000e+00
[Best] Ep 0 | step 4454 | reward 0.0 (prev -inf)
10000/10000 [==============================] - 20s 2ms/step - reward: 0.0000e+00
2 episodes - episode_reward: 0.000 [0.000, 0.000] - lives: 0.000 - episode_frame_number: 6122.994 - frame_number: 14950.181

Interval 2 (10000 steps performed)
10000/10000 [==============================] - 20s 2ms/step - reward: 0.0000e+00
2 episodes - episode_reward: 0.000 [0.000, 0.000] - lives: 0.000 - episode_frame_number: 6481.411 - frame_number: 44911.824

Interval 3 (20000 steps performed)
10000/10000 [==============================] - 20s 2ms/step - reward: 0.0000e+00
2 episodes - episode_reward: 0.000 [0.000, 0.000] - lives: 0.000 - episode_frame_number: 6832.256 - frame_number: 74880.538

Interval 4 (30000 steps performed)
10000/10000 [==============================] - 20s 2ms/step - reward: 0.0000e+00
3 episodes - episode_reward: 0.000 [0.000, 0.000] - lives: 0.000 - epi

2026-06-28 15:20:09.870502: W tensorflow/c/c_api.cc:305] Operation '{name:'dense_1_1/BiasAdd' id:249 op device:{requested: '', assigned: ''} def:{{{node dense_1_1/BiasAdd}} = BiasAdd[T=DT_FLOAT, _has_manual_control_dependencies=true, data_format="NHWC"](dense_1_1/MatMul, dense_1_1/BiasAdd/ReadVariableOp)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
2026-06-28 15:20:09.943110: W tensorflow/c/c_api.cc:305] Operation '{name:'loss_3/AddN' id:491 op device:{requested: '', assigned: ''} def:{{{node loss_3/AddN}} = AddN[N=2, T=DT_FLOAT, _has_manual_control_dependencies=true](loss_3/mul, loss_3/mul_1)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
2026-06-28 15:20:0

10000/10000 [==============================] - 123s 12ms/step - reward: 0.0000e+00
2 episodes - episode_reward: 0.000 [0.000, 0.000] - loss: 0.000 - mae: 0.075 - mean_q: 0.090 - mean_eps: 0.951 - lives: 0.000 - episode_frame_number: 6482.000 - frame_number: 164853.533

Interval 7 (60000 steps performed)
10000/10000 [==============================] - 124s 12ms/step - reward: 0.0000e+00
2 episodes - episode_reward: 0.000 [0.000, 0.000] - loss: 0.000 - mae: 0.084 - mean_q: 0.097 - mean_eps: 0.942 - lives: 0.000 - episode_frame_number: 6854.765 - frame_number: 194782.932

Interval 8 (70000 steps performed)
10000/10000 [==============================] - 124s 12ms/step - reward: 0.0000e+00
3 episodes - episode_reward: 0.000 [0.000, 0.000] - loss: 0.000 - mae: 0.085 - mean_q: 0.099 - mean_eps: 0.933 - lives: 0.000 - episode_frame_number: 7153.855 - frame_number: 224871.614

Interval 9 (80000 steps performed)
10000/10000 [==============================] - 124s 12ms/step - reward: 0.0000e+00
2 

I0000 00:00:1782653431.273321 3808053 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1782653431.273338 3808053 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


10000/10000 [==============================] - 126s 13ms/step - reward: 0.0000e+00
2 episodes - episode_reward: 0.000 [0.000, 0.000] - loss: 0.000 - mae: 0.090 - mean_q: 0.104 - mean_eps: 0.906 - lives: 0.000 - episode_frame_number: 6894.973 - frame_number: 314962.608

Interval 12 (110000 steps performed)
10000/10000 [==============================] - 126s 13ms/step - reward: 0.0000e+00
3 episodes - episode_reward: 0.000 [0.000, 0.000] - loss: 0.000 - mae: 0.091 - mean_q: 0.105 - mean_eps: 0.897 - lives: 0.000 - episode_frame_number: 7021.279 - frame_number: 345052.895

Interval 13 (120000 steps performed)
10000/10000 [==============================] - 126s 13ms/step - reward: 0.0000e+00
2 episodes - episode_reward: 0.000 [0.000, 0.000] - loss: 0.000 - mae: 0.093 - mean_q: 0.108 - mean_eps: 0.888 - lives: 0.000 - episode_frame_number: 6185.153 - frame_number: 375099.277

Interval 14 (130000 steps performed)
10000/10000 [==============================] - 127s 13ms/step - reward: 0.0000e

: 

### 1.4 Evaluación del agente

Carga los mejores pesos guardados durante el entrenamiento y ejecuta `NB_TEST_EPISODES = 100`
episodios sin exploración (`EPS_TEST = 0.05`).

**Criterio de éxito**: el agente supera el primer día de carrera (≥ `DAY1_TARGET = 200` coches
adelantados) de forma consistente en los 100 episodios.

In [ ]:
# Carga los pesos del mejor episodio
# TF guarda en formato checkpoint: base.h5f.index + base.h5f.data-* -> buscar por .index
best_files = sorted((WEIGHTS_FOLDER / "final").glob(f"dqn_{ENV_NAME}_best_step*.h5f.index"))
weights_file = str(best_files[-1])[: -len(".index")]  # quita el sufijo .index
print(f"Cargando: {weights_file}")

dqn.load_weights(weights_file)

# gym 0.25 requiere render_mode en make(), no en render() -> env separado para visualización
visualize = False
env_vis = gym.make(ENV_NAME, render_mode="human") if visualize else gym.make(ENV_NAME)
dqn.test(env_vis, nb_episodes=NB_TEST_EPISODES, visualize=False)
env_vis.close()

Cargando: /Users/alejandro/projects/msc_ia/AR/trabajo/ar_proyecto_grupal_02/weights/final/dqn_Enduro-v0_best_step1509345.h5f
Testing for 100 episodes ...


I0000 00:00:1782683854.063605 4220352 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1782683854.063623 4220352 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-06-28 23:57:34.078825: W tensorflow/c/c_api.cc:305] Operation '{name:'total_2/Assign' id:380 op device:{requested: '', assigned: ''} def:{{{node total_2/Assign}} = AssignVariableOp[_has_manual_control_dependencies=true, dtype=DT_FLOAT, validate_shape=false](total_2, total_2/Initializer/zeros)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
/Users/alejandro/miniconda3/envs/ar_env/lib/python3.10/

Episode 1: reward: 0.000, steps: 4425
Episode 2: reward: 0.000, steps: 4457
Episode 3: reward: 0.000, steps: 4429
Episode 4: reward: 0.000, steps: 4430
Episode 5: reward: 0.000, steps: 4447
Episode 6: reward: 0.000, steps: 4431
Episode 7: reward: 0.000, steps: 4475
Episode 8: reward: 0.000, steps: 4462
Episode 9: reward: 0.000, steps: 4428
Episode 10: reward: 0.000, steps: 4436
Episode 11: reward: 0.000, steps: 4435
Episode 12: reward: 0.000, steps: 4454
Episode 13: reward: 0.000, steps: 4461
Episode 14: reward: 0.000, steps: 4436
Episode 15: reward: 0.000, steps: 4419
Episode 16: reward: 1.000, steps: 4414
Episode 17: reward: 0.000, steps: 4466
Episode 18: reward: 5.000, steps: 4438
Episode 19: reward: 0.000, steps: 4453
Episode 20: reward: 0.000, steps: 4487
Episode 21: reward: 10.000, steps: 4433
Episode 22: reward: 0.000, steps: 4445
Episode 23: reward: 0.000, steps: 4455
Episode 24: reward: 6.000, steps: 4432
Episode 25: reward: 11.000, steps: 4467
Episode 26: reward: 0.000, steps

## 2. Mejoras y experimentos

### 2.1 Mejora 1

#### 2.1.1 Implementación de la red neuronal

#### 2.1.2 Implementación del agente (DQN)

#### 2.1.3 Entrenamiento del agente

#### 2.1.4 Evaluación del agente

### 2.2 Mejora 2

#### 2.2.1 Implementación de la red neuronal

#### 2.2.2 Implementación del agente (DQN)

#### 2.2.3 Entrenamiento del agente

#### 2.2.4 Evaluación del agente

### 2.3 Mejora 3

#### 2.3.1 Implementación de la red neuronal

#### 2.3.2 Implementación del agente (DQN)

#### 2.3.3 Entrenamiento del agente

#### 2.3.4 Evaluación del agente

## 3. Conclusiones
Justificación de los parámetros seleccionados y de los resultados obtenidos

